# SciFact Retrieval Benchmark

BM25 → Dense → Hybrid RRF → Cross-Encoder reranking.
Reusable code is kept in `src/`.

## Setup

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
from pathlib import Path
import sys, pandas as pd

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

from src.retrieval import *
from src.evaluation import *

## Load SciFact

In [ ]:
corpus, queries, qrels = load_scifact(data_dir=str(ROOT / "datasets"))
doc_ids, documents = prepare_documents(corpus)

print(len(corpus), len(queries), len(qrels))

## BM25

In [ ]:
bm25 = build_bm25(documents)

def bm25_fn(query, k=10):
    return retrieve_bm25(query, bm25, doc_ids, k)

## Dense retrieval

In [ ]:
dense_model = load_dense_model("intfloat/multilingual-e5-base")

document_embeddings, doc_ids = build_or_load_document_embeddings(
    dense_model,
    documents,
    doc_ids,
    cache_dir=ROOT / "cache"
)

def dense_fn(query, k=10):
    return retrieve_dense(query, dense_model, document_embeddings, doc_ids, k)

## Hybrid RRF

In [ ]:
def hybrid_fn(query, k=10):
    return retrieve_hybrid(
        query, bm25, dense_model, document_embeddings, doc_ids,
        k=k, candidate_k=100, rrf_k=60
    )

## Cross-Encoder reranking

In [ ]:
reranker = load_reranker("BAAI/bge-reranker-v2-m3")

def reranked_fn(query, k=10):
    return retrieve_reranked(
        query, corpus, bm25, dense_model, document_embeddings, doc_ids, reranker,
        k=k, candidate_k=20, hybrid_candidate_k=100
    )

## Evaluate all methods

In [ ]:
results = pd.DataFrame(
    [
        evaluate_retriever(bm25_fn, queries, qrels),
        evaluate_retriever(dense_fn, queries, qrels),
        evaluate_retriever(hybrid_fn, queries, qrels),
        evaluate_retriever(reranked_fn, queries, qrels),
    ],
    index=["BM25", "Dense", "Hybrid RRF", "Hybrid + Cross-Encoder"]
)
results

## Recorded results

In [ ]:
pd.read_csv(ROOT / "results" / "metrics.csv")

## Error-analysis candidate search

In [ ]:
print("BM25 fail → Dense success:",
      len(find_bm25_fail_dense_success(queries, qrels, bm25_fn, dense_fn)))

print("Dense fail → BM25 success:",
      len(find_dense_fail_bm25_success(queries, qrels, bm25_fn, dense_fn)))

print("Dense fail → Hybrid success:",
      len(find_dense_fail_hybrid_success(queries, qrels, dense_fn, hybrid_fn)))

## Find cases improved by the Cross-Encoder

In [ ]:
reranker_cases = find_reranker_improvements_batched(
    queries=queries,
    qrels=qrels,
    corpus=corpus,
    retrieve_hybrid_fn=hybrid_fn,
    reranker=reranker,
    candidate_k=20,
    batch_size=64,
)

print("Number of improvements:", len(reranker_cases))
reranker_cases[:20]

## Selected qualitative cases
See `results/error_analysis.md`.

## Next
1. add one reranker-improvement case
2. multilingual evaluation
3. agentic RAG extension